# Multi-regions synthesis — overshoot by region (CBA vs PBA)

"Overshoot / Safe Operating Space" figure for the 2019 geographies
(France / EU27 / World / regions, `2019_*` folders). For each scenario, 2
bubbles per LP:
- solid bubble: **consumption-based** accounting (CBA, `d_cba.pkl`)
- hatched bubble: **production-based** accounting (PBA, `F_x_dom.pkl`, domestic part only)

Generates `figures/multi-regions_synthesis/overshoot_safe_operating_space_by_region.png`
(`create_multi_regions_synthesis`) and the Excel workbook of its data (same name, `.xlsx`).

In [ ]:
# ============================ PARAMETERS ============================
SCENARIO_GROUP = "REGIONS"        # a 2019 display group of scenario_groups.json
SHARING_PRINCIPLE = "CTR"         # national budget = world budget x share in budget_shares.xlsx:
                                  #   "EPC" (equal per capita) or "CTR" (capability to reduce)
SENSITIVITY = True                # True: min/max segment of the sharing principle on each bubble
SHOW_PBA = True                   # True: hatched production-based (PBA) bubble next to the CBA one
DISPLAY_BOUNDS = True             # True: Lower/High-Risk Boundary lines shown in addition to the Safe Boundary
SCALE_UB = True                   # True: scale rescaled per LP so that the High-Risk Boundary is aligned; False: common nL scale
X_MAIN_MAX = 6                    # right bound of the main axis, in multiples of the Safe Boundary
SHOW_VALUE_OVERSHOOT = True       # True: one "nL" axis per bar; False: a single axis at the top
TITLE = f"Overshoot relative to the Safe Operating Space, by region ({SHARING_PRINCIPLE})"
DPI = 300                         # resolution of the saved .png
# ====================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

notebooks_dir = Path.cwd()
while not (notebooks_dir / "planefr_lib").exists() and notebooks_dir != notebooks_dir.parent:
    notebooks_dir = notebooks_dir.parent
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

import matplotlib.pyplot as plt

from planefr_lib import config, io, processing
from planefr_lib.plot_overshoot import create_multi_regions_synthesis

# Folder of the figures (and of their Excel workbooks): figures/<NOTEBOOK_NAME>/<SCENARIO_GROUP>/
NOTEBOOK_NAME = "multi-regions_synthesis"
figures_dir = config.figure_dir(NOTEBOOK_NAME, SCENARIO_GROUP)

## 1. Loading the data and per-capita computation

In [ ]:
cf_df = io.load_characterization_factors(SCENARIO_GROUP)
boundaries_df = io.load_boundaries(SCENARIO_GROUP)
pop_df = io.load_population_df()

# Unlike the other notebooks: geographies rather than scenarios (only the folders of SCENARIO_GROUP).
scenario_folders = config.get_scenario_folders(exclude_2019=True, scenario_group=SCENARIO_GROUP)
scenario_names = [f.name for f in scenario_folders]
print(f"Scenarios found ({len(scenario_folders)}): {scenario_names}")

In [3]:
all_scenarios_data = []
for scenario_folder in scenario_folders:
    print(f"Processing: {scenario_folder.name}")
    data = processing.process_scenario_per_capita(scenario_folder, cf_df, boundaries_df, pop_df)
    print(f"  Indicators computed: {list(data.keys())}")
    all_scenarios_data.append(data)

print(f"\n{len(all_scenarios_data)} scenarios processed.")

Processing: 2019_EU27
  Indicators computed: ['N surplus', 'P surplus', 'Blue water consumption', 'Cropland use', 'Agricultural land', 'GHG emissions', 'Biodiversity loss - PDF', 'Biodiversity loss - HANPP', 'Raw material consumption']
Processing: 2019_FR
  Indicators computed: ['N surplus', 'P surplus', 'Blue water consumption', 'Cropland use', 'Agricultural land', 'GHG emissions', 'Biodiversity loss - PDF', 'Biodiversity loss - HANPP', 'Raw material consumption']
Processing: 2019_W
  Indicators computed: ['N surplus', 'P surplus', 'Blue water consumption', 'Cropland use', 'Agricultural land', 'GHG emissions', 'Biodiversity loss - PDF', 'Biodiversity loss - HANPP', 'Raw material consumption']
Processing: 2019_WF
  Indicators computed: ['N surplus', 'P surplus', 'Blue water consumption', 'Cropland use', 'Agricultural land', 'GHG emissions', 'Biodiversity loss - PDF', 'Biodiversity loss - HANPP', 'Raw material consumption']

4 scenarios processed.


## 2. Overshoot figure in absolute values, by country/region (EPC budget)

Variant in absolute values (nothing is divided by the population): the budget
of each scenario is the world budget multiplied by its share in
`budget_shares.xlsx` (UN EPC column, row of the EXIOBASE code deduced from the
folder name). See `create_multi_regions_synthesis`.

In [ ]:
shares_df = io.load_budget_shares_df()
data_absolute = [processing.process_scenario_absolute(f, cf_df, boundaries_df) for f in scenario_folders]
output_path = figures_dir / "overshoot_safe_operating_space_by_region.png"

fig = create_multi_regions_synthesis(
    all_scenarios_data=data_absolute, boundaries_df=boundaries_df, shares_df=shares_df,
    scenario_names=scenario_names, x_main_max=X_MAIN_MAX, display_bounds=DISPLAY_BOUNDS,
    show_pba=SHOW_PBA,
    show_value_overshoot=SHOW_VALUE_OVERSHOOT, title=TITLE,
    sharing_principle=SHARING_PRINCIPLE, sensitivity=SENSITIVITY, scale_UB=SCALE_UB,
    pop_df=pop_df,  # sensitivity of W: per-capita limit depending on the population period
    excel_path=output_path.with_suffix(".xlsx"))
fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
print(f"Figure saved: {output_path}")
plt.show()